# SofaScore Team Form Extractor

This is the second notebook in the team-form workflow. It reads `bundesliga_teams.json`, opens one reusable `undetected_chromedriver` Chrome instance, and retrieves up to two pages of SofaScore event history for every team.

For each team, it calculates five-match overall form and Bundesliga-only form, keeps the detailed matches used in both calculations, and orders each form from oldest to newest. Each run creates a separate, timestamped `team_form_*.json` file in `outputs/sofascore/team_form`. The execution timestamp is also retained as the JSON object's top-level key.

Run `01_extract_bundesliga_teams.ipynb` first to generate the required team input file.


In [1]:
# Resolve the project root and import authoritative data locations.
import sys
from pathlib import Path


# Handle project root for reuse in the workflow.
def _locate_project_root() -> Path:
    starts = []
    vscode_notebook = globals().get("__vsc_ipynb_file__")
    if isinstance(vscode_notebook, str) and vscode_notebook.strip():
        starts.append(Path(vscode_notebook).expanduser().resolve().parent)
    starts.append(Path.cwd().resolve())

    checked = set()
    # Process each available item while preserving the current workflow state.
    for start in starts:
        # Process each available item while preserving the current workflow state.
        for candidate in (start, *start.parents):
            if candidate in checked:
                continue
            checked.add(candidate)
            if (candidate / "project_paths.py").is_file():
                return candidate
    raise FileNotFoundError(
        "Could not locate project_paths.py. Start Jupyter from the Kickbase "
        "project root or open this notebook from within that project."
    )


# Set workflow configuration value: _PROJECT_ROOT.
_PROJECT_ROOT = _locate_project_root()
if str(_PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(_PROJECT_ROOT))

from project_paths import (
    SOFASCORE_REFERENCE_DIR,
    SOFASCORE_TEAM_FORM_DIR,
    ensure_directory,
)


In [2]:
# 1. Imports
import json
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

# Handle expected failures with a clear, actionable message.
try:
    import pandas as pd
    import undetected_chromedriver as uc
    from bs4 import BeautifulSoup
    from IPython.display import display
    from selenium.common.exceptions import TimeoutException, WebDriverException
    from selenium.webdriver.common.by import By
    from selenium.webdriver.support import expected_conditions as EC
    from selenium.webdriver.support.ui import WebDriverWait
except ImportError as exc:
    raise ImportError(
        "Required packages are missing. Install undetected-chromedriver, "
        "selenium, beautifulsoup4, and pandas in this Jupyter kernel, "
        "then restart the kernel."
    ) from exc


In [3]:
# 2. Parameters, Paths, and Snapshot Key
CHROME_MAJOR_VERSION = None  # Let undetected-chromedriver auto-detect Chrome.
# Set workflow configuration value: HEADLESS.
HEADLESS = False
# Set workflow configuration value: PAGE_LOAD_TIMEOUT_SECONDS.
PAGE_LOAD_TIMEOUT_SECONDS = 30
# Set workflow configuration value: WAIT_TIMEOUT_SECONDS.
WAIT_TIMEOUT_SECONDS = 20
# Set workflow configuration value: MAX_PAGES.
MAX_PAGES = 1
# Set workflow configuration value: FORM_MATCH_COUNT.
FORM_MATCH_COUNT = 5
# Set workflow configuration value: BUNDESLIGA_UNIQUE_TOURNAMENT_ID.
BUNDESLIGA_UNIQUE_TOURNAMENT_ID = 35
# Set workflow configuration value: TEAM_EVENTS_URL_TEMPLATE.
TEAM_EVENTS_URL_TEMPLATE = (
    "https://www.sofascore.com/api/v1/team/{team_id}/events/last/{page}"
)

# Process each available item while preserving the current workflow state.
for setting_name, setting_value in {
    "PAGE_LOAD_TIMEOUT_SECONDS": PAGE_LOAD_TIMEOUT_SECONDS,
    "WAIT_TIMEOUT_SECONDS": WAIT_TIMEOUT_SECONDS,
    "MAX_PAGES": MAX_PAGES,
    "FORM_MATCH_COUNT": FORM_MATCH_COUNT,
}.items():
    # Validate the input before continuing with later processing.
    if not isinstance(setting_value, int) or isinstance(setting_value, bool) or setting_value < 1:
        raise ValueError(f"{setting_name} must be a positive integer.")

# Validate the input before continuing with later processing.
if FORM_MATCH_COUNT != 5:
    raise ValueError("FORM_MATCH_COUNT must remain 5 for five-match form.")

teams_path = SOFASCORE_REFERENCE_DIR / "bundesliga_teams.json"
execution_datetime = datetime.now().astimezone()
execution_timestamp = execution_datetime.isoformat(timespec="microseconds")
filename_timestamp = execution_datetime.strftime("%Y-%m-%d_%H-%M-%S_%f%z")
snapshot_output_path = ensure_directory(SOFASCORE_TEAM_FORM_DIR) / f"team_form_{filename_timestamp}.json"

print(f"Execution timestamp: {execution_timestamp}")
print(f"Team input file: {teams_path}")
print(f"Snapshot output file: {snapshot_output_path}")


Execution timestamp: 2026-09-04T10:43:32.643242+02:00
Team input file: C:\kickbase project\outputs\sofascore\reference\bundesliga_teams.json
Snapshot output file: C:\kickbase project\outputs\sofascore\team_form\team_form_2026-09-04_10-43-32_643242+0200.json


In [4]:
# 3. Load Bundesliga Teams
try:
    raw_teams = json.loads(teams_path.read_text(encoding="utf-8"))
except FileNotFoundError as exc:
    raise FileNotFoundError(
        f"Team file not found: {teams_path}. Run notebooks/01_fixtures_and_teams/03_build_bundesliga_team_reference.ipynb first."
    ) from exc
except UnicodeDecodeError as exc:
    raise ValueError(f"Team file is not valid UTF-8: {teams_path}") from exc
except json.JSONDecodeError as exc:
    raise ValueError(
        f"Team file is not valid JSON (line {exc.lineno}, column {exc.colno}): "
        f"{teams_path}"
    ) from exc
except OSError as exc:
    raise OSError(f"Could not read team file {teams_path}: {exc}") from exc

# Validate the input before continuing with later processing.
if not isinstance(raw_teams, dict):
    raise ValueError("The team JSON must be an object keyed by numeric team ID.")

teams: dict[int, dict[str, Any]] = {}
# Process each available item while preserving the current workflow state.
for team_key, team_record in raw_teams.items():
    # Validate the input before continuing with later processing.
    if not isinstance(team_record, dict):
        raise ValueError(f"Team entry {team_key!r} must be a JSON object.")

    team_id = team_record.get("team_id")
    team_name = team_record.get("team")
    # Validate the input before continuing with later processing.
    if not isinstance(team_id, int) or isinstance(team_id, bool) or team_id < 1:
        raise ValueError(f"Team entry {team_key!r} has no valid positive team_id.")
    # Validate the input before continuing with later processing.
    if str(team_id) != str(team_key):
        raise ValueError(
            f"Team key {team_key!r} does not match its team_id {team_id}."
        )
    # Validate the input before continuing with later processing.
    if not isinstance(team_name, str) or not team_name.strip():
        raise ValueError(f"Team entry {team_key!r} has no valid team name.")
    # Validate the input before continuing with later processing.
    if team_id in teams:
        raise ValueError(f"Duplicate team ID in team file: {team_id}.")

    teams[team_id] = {"team": team_name.strip()}

print(f"Loaded and validated {len(teams)} unique team(s).")


Loaded and validated 18 unique team(s).


In [5]:
# 4. Start Chrome
options = uc.ChromeOptions()
options.add_argument("--disable-gpu")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--window-size=1920,1080")
if HEADLESS:
    options.add_argument("--headless=new")

driver = None
# Handle expected failures with a clear, actionable message.
try:
    driver = uc.Chrome(
        options=options,
                use_subprocess=True,
    )
    driver.set_page_load_timeout(PAGE_LOAD_TIMEOUT_SECONDS)
except Exception as exc:
    if driver is not None:
        # Handle expected failures with a clear, actionable message.
        try:
            driver.quit()
        except Exception:
            pass
        driver = None
    raise RuntimeError(
        f"Could not initialize undetected Chrome with major version "
        f"{CHROME_MAJOR_VERSION}. Adjust CHROME_MAJOR_VERSION if needed. "
        f"Original error: {exc}"
    ) from exc

print(
    f"One reusable undetected Chrome instance is ready "
    f"(major version {CHROME_MAJOR_VERSION}, headless={HEADLESS})."
)


One reusable undetected Chrome instance is ready (major version None, headless=False).


In [6]:
# 5. SofaScore API Helper
class SofaScorePageError(Exception):
    """Raised when Chrome cannot return a usable SofaScore event page."""


# Retrieve team events page for reuse in the workflow.
def fetch_team_events_page(
    team_id: int,
    page: int,
) -> tuple[list[dict[str, Any]], bool]:
    # Validate the input before continuing with later processing.
    if driver is None:
        raise SofaScorePageError("Chrome is not initialized.")

    url = TEAM_EVENTS_URL_TEMPLATE.format(team_id=team_id, page=page)
    print(f"    Loading page {page}: {url}")
    # Handle expected failures with a clear, actionable message.
    try:
        driver.get(url)
    except TimeoutException as exc:
        raise SofaScorePageError(
            f"Timed out after {PAGE_LOAD_TIMEOUT_SECONDS} seconds loading {url}."
        ) from exc
    except WebDriverException as exc:
        raise SofaScorePageError(f"Chrome could not load {url}: {exc}") from exc

    # Handle expected failures with a clear, actionable message.
    try:
        WebDriverWait(driver, WAIT_TIMEOUT_SECONDS).until(
            EC.presence_of_element_located((By.TAG_NAME, "pre"))
        )
    except TimeoutException as exc:
        raise SofaScorePageError(
            f"No <pre> element appeared within {WAIT_TIMEOUT_SECONDS} seconds for {url}."
        ) from exc
    except WebDriverException as exc:
        raise SofaScorePageError(
            f"Chrome could not inspect the rendered response for {url}: {exc}"
        ) from exc

    soup = BeautifulSoup(driver.page_source, "html.parser")
    pre_tag = soup.find("pre")
    # Validate the input before continuing with later processing.
    if pre_tag is None:
        raise SofaScorePageError(f"Rendered page contains no <pre> element: {url}")

    response_text = pre_tag.get_text().strip()
    # Validate the input before continuing with later processing.
    if not response_text:
        raise SofaScorePageError(f"The <pre> element is empty: {url}")

    # Handle expected failures with a clear, actionable message.
    try:
        payload = json.loads(response_text)
    except json.JSONDecodeError as exc:
        raise SofaScorePageError(
            f"Invalid JSON for {url} (line {exc.lineno}, column {exc.colno})."
        ) from exc

    # Validate the input before continuing with later processing.
    if not isinstance(payload, dict):
        raise SofaScorePageError(f"SofaScore response is not a JSON object: {url}")
    events = payload.get("events")
    # Validate the input before continuing with later processing.
    if not isinstance(events, list):
        raise SofaScorePageError(
            f"SofaScore response has no valid events list: {url}"
        )

    has_next_page = payload.get("hasNextPage")
    if not isinstance(has_next_page, bool):
        print("    Warning: hasNextPage is missing or invalid; treating it as false.")
        has_next_page = False

    return events, has_next_page


In [7]:
# 6. Result Parser
def nested_get(mapping: Any, *keys: str) -> Any:
    current = mapping
    # Process each available item while preserving the current workflow state.
    for key in keys:
        if not isinstance(current, dict):
            return None
        current = current.get(key)
    return current


# Check whether number for reuse in the workflow.
def is_number(value: Any) -> bool:
    return isinstance(value, (int, float)) and not isinstance(value, bool)


# Parse and validate finished event for reuse in the workflow.
def parse_finished_event(
    event: dict[str, Any],
    team_id: int,
) -> dict[str, Any] | None:
    if not isinstance(event, dict):
        print("    Warning: skipped a non-object event.")
        return None

    event_id = event.get("id")
    if not isinstance(event_id, int) or isinstance(event_id, bool):
        print("    Warning: skipped an event with no valid numeric ID.")
        return None
    if nested_get(event, "status", "type") != "finished":
        return None

    home_team_id = nested_get(event, "homeTeam", "id")
    away_team_id = nested_get(event, "awayTeam", "id")
    if (
        not isinstance(home_team_id, int)
        or isinstance(home_team_id, bool)
        or not isinstance(away_team_id, int)
        or isinstance(away_team_id, bool)
    ):
        print(f"    Warning: skipped event {event_id}; team IDs are missing or invalid.")
        return None

    # Choose the appropriate path for the current data state.
    if team_id == home_team_id:
        analysed_team_is_home = True
    # Choose the appropriate path for the current data state.
    elif team_id == away_team_id:
        analysed_team_is_home = False
    else:
        print(
            f"    Warning: skipped event {event_id}; team ID {team_id} "
            "matches neither side."
        )
        return None

    home_score = nested_get(event, "homeScore", "normaltime")
    away_score = nested_get(event, "awayScore", "normaltime")
    if not is_number(home_score) or not is_number(away_score):
        print(
            f"    Warning: skipped event {event_id}; normal-time scores are missing."
        )
        return None

    timestamp = event.get("startTimestamp")
    if not is_number(timestamp):
        print(f"    Warning: skipped event {event_id}; startTimestamp is missing.")
        return None
    # Handle expected failures with a clear, actionable message.
    try:
        readable_date = datetime.fromtimestamp(timestamp, tz=timezone.utc).isoformat()
    except (OverflowError, OSError, ValueError) as exc:
        print(f"    Warning: skipped event {event_id}; invalid timestamp ({exc}).")
        return None

    analysed_score = home_score if analysed_team_is_home else away_score
    opponent_score = away_score if analysed_team_is_home else home_score
    # Choose the appropriate path for the current data state.
    if analysed_score > opponent_score:
        result = "W"
    # Choose the appropriate path for the current data state.
    elif analysed_score < opponent_score:
        result = "L"
    else:
        result = "D"

    return {
        "match_id": event_id,
        "date": readable_date,
        "timestamp": timestamp,
        "competition": nested_get(event, "tournament", "name"),
        "unique_tournament_id": nested_get(
            event, "tournament", "uniqueTournament", "id"
        ),
        "home_team": nested_get(event, "homeTeam", "name"),
        "home_team_id": home_team_id,
        "away_team": nested_get(event, "awayTeam", "name"),
        "away_team_id": away_team_id,
        "home_score": home_score,
        "away_score": away_score,
        "result": result,
    }


In [8]:
# 7. Team Form Function
def select_recent_matches(
    candidates: list[dict[str, Any]],
) -> tuple[list[dict[str, Any]], str]:
    latest_matches = sorted(
        candidates,
        key=lambda match: match["timestamp"],
        reverse=True,
    )[:FORM_MATCH_COUNT]
    selected_matches = sorted(
        latest_matches,
        key=lambda match: match["timestamp"],
    )
    form = "".join(match["result"] for match in selected_matches)
    return selected_matches, form


# Collect team form for reuse in the workflow.
def collect_team_form(team_id: int, team_name: str) -> dict[str, Any]:
    overall_candidates: list[dict[str, Any]] = []
    bundesliga_candidates: list[dict[str, Any]] = []
    seen_event_ids: set[int] = set()

    # Process each available item while preserving the current workflow state.
    for page in range(MAX_PAGES):
        # Handle expected failures with a clear, actionable message.
        try:
            events, has_next_page = fetch_team_events_page(team_id, page)
        except SofaScorePageError as exc:
            print(f"    Warning: stopped pagination for {team_name}: {exc}")
            break

        if not events:
            print(f"    No events returned on page {page}; history has ended.")
            break

        # Process each available item while preserving the current workflow state.
        for event in events:
            event_id = event.get("id") if isinstance(event, dict) else None
            if isinstance(event_id, int) and not isinstance(event_id, bool):
                if event_id in seen_event_ids:
                    continue
                seen_event_ids.add(event_id)

            record = parse_finished_event(event, team_id)
            if record is None:
                continue
            overall_candidates.append(record)

            if record["unique_tournament_id"] == BUNDESLIGA_UNIQUE_TOURNAMENT_ID:
                unique_tournament = nested_get(
                    event, "tournament", "uniqueTournament"
                )
                unique_name = (
                    unique_tournament.get("name")
                    if isinstance(unique_tournament, dict)
                    else None
                )
                unique_slug = (
                    unique_tournament.get("slug")
                    if isinstance(unique_tournament, dict)
                    else None
                )
                if unique_name != "Bundesliga" or unique_slug != "bundesliga":
                    print(
                        f"    Warning: event {record['match_id']} has Bundesliga ID "
                        f"{BUNDESLIGA_UNIQUE_TOURNAMENT_ID} but unexpected "
                        f"name/slug values: {unique_name!r}, {unique_slug!r}."
                    )
                bundesliga_candidates.append(record)

        if (
            len(overall_candidates) >= FORM_MATCH_COUNT
            and len(bundesliga_candidates) >= FORM_MATCH_COUNT
        ):
            break
        if not has_next_page:
            print(f"    SofaScore reports no page after page {page}.")
            break
    else:
        print(f"    Reached MAX_PAGES={MAX_PAGES} for {team_name}.")

    overall_matches, overall_form = select_recent_matches(overall_candidates)
    bundesliga_matches, bundesliga_form = select_recent_matches(
        bundesliga_candidates
    )
    if len(overall_matches) < FORM_MATCH_COUNT:
        print(
            f"    Warning: {team_name} has only {len(overall_matches)} valid "
            f"overall match(es); expected {FORM_MATCH_COUNT}."
        )
    if len(bundesliga_matches) < FORM_MATCH_COUNT:
        print(
            f"    Warning: {team_name} has only {len(bundesliga_matches)} valid "
            f"Bundesliga match(es); expected {FORM_MATCH_COUNT}."
        )

    return {
        "team": team_name,
        "overall_form": overall_form,
        "bundesliga_form": bundesliga_form,
        "overall_matches": overall_matches,
        "bundesliga_matches": bundesliga_matches,
    }


In [9]:
# 8. Process All Teams
form_results: dict[int, dict[str, Any]] = {}
total_teams = len(teams)

# Process each available item while preserving the current workflow state.
for team_number, (team_id, team_info) in enumerate(teams.items(), start=1):
    team_name = team_info["team"]
    print(f"[{team_number}/{total_teams}] {team_name} (team_id={team_id})")
    # Handle expected failures with a clear, actionable message.
    try:
        form_results[team_id] = collect_team_form(team_id, team_name)
    except Exception as exc:
        print(
            f"    Unexpected team error: {type(exc).__name__}: {exc}. "
            "Continuing with the remaining teams."
        )
        form_results[team_id] = {
            "team": team_name,
            "overall_form": "",
            "bundesliga_form": "",
            "overall_matches": [],
            "bundesliga_matches": [],
        }

print("Finished processing all teams.")


[1/18] FC Bayern München (team_id=2672)
    Loading page 0: https://www.sofascore.com/api/v1/team/2672/events/last/0
[2/18] VfB Stuttgart (team_id=2677)
    Loading page 0: https://www.sofascore.com/api/v1/team/2677/events/last/0
[3/18] 1. FC Köln (team_id=2671)
    Loading page 0: https://www.sofascore.com/api/v1/team/2671/events/last/0
[4/18] TSG Hoffenheim (team_id=2569)
    Loading page 0: https://www.sofascore.com/api/v1/team/2569/events/last/0
[5/18] 1. FC Union Berlin (team_id=2547)
    Loading page 0: https://www.sofascore.com/api/v1/team/2547/events/last/0
[6/18] Eintracht Frankfurt (team_id=2674)
    Loading page 0: https://www.sofascore.com/api/v1/team/2674/events/last/0
[7/18] 1. FSV Mainz 05 (team_id=2556)
    Loading page 0: https://www.sofascore.com/api/v1/team/2556/events/last/0
[8/18] SC Paderborn 07 (team_id=2561)
    Loading page 0: https://www.sofascore.com/api/v1/team/2561/events/last/0
    Reached MAX_PAGES=1 for SC Paderborn 07.
[9/18] RB Leipzig (team_id=36360)


In [10]:
# 9. Final Form DataFrame
summary_rows = [
    {
        "team_id": team_id,
        "team": result["team"],
        "overall_form": result["overall_form"],
        "overall_match_count": len(result["overall_matches"]),
        "bundesliga_form": result["bundesliga_form"],
        "bundesliga_match_count": len(result["bundesliga_matches"]),
    }
    for team_id, result in form_results.items()
]
form_summary_df = pd.DataFrame(summary_rows)
display(form_summary_df)


,team_id,team,overall_form,overall_match_count,bundesliga_form,bundesliga_match_count
0,2672,FC Bayern München,WWWWW,5,WDWWW,5
1,2677,VfB Stuttgart,DWLWL,5,DDWDL,5
2,2671,1. FC Köln,WDWWW,5,LDLLW,5
3,2569,TSG Hoffenheim,LLDWL,5,WDWLL,5
4,2547,1. FC Union Berlin,DWLWD,5,LDWWD,5
5,2674,Eintracht Frankfurt,WWLWD,5,DLLDD,5
6,2556,1. FSV Mainz 05,DWWWD,5,LWLWD,5
7,2561,SC Paderborn 07,WWDWD,5,DDD,3
8,36360,RB Leipzig,WLLWW,5,WLWLW,5
9,2527,Borussia M'gladbach,WWWWL,5,DWLWL,5


In [11]:
# 10. Detailed Histories
history_columns = [
    "date", "competition", "home_team", "away_team",
    "home_score", "away_score", "result", "match_id",
    "timestamp", "unique_tournament_id",
]

# Process each available item while preserving the current workflow state.
for team_id, result in form_results.items():
    print(f"\n{result['team']} (team_id={team_id})")
    print(f"Overall form: {result['overall_form'] or '(no valid matches)'}")
    # Choose the appropriate path for the current data state.
    if result["overall_matches"]:
        display(pd.DataFrame(result["overall_matches"]).reindex(columns=history_columns))
    else:
        print("No valid overall matches were collected.")

    print(f"Bundesliga form: {result['bundesliga_form'] or '(no valid matches)'}")
    # Choose the appropriate path for the current data state.
    if result["bundesliga_matches"]:
        display(
            pd.DataFrame(result["bundesliga_matches"]).reindex(
                columns=history_columns
            )
        )
    else:
        print("No valid Bundesliga matches were collected.")



FC Bayern München (team_id=2672)
Overall form: WWWWW


,date,competition,home_team,away_team,home_score,away_score,result,match_id,timestamp,unique_tournament_id
0,2026-08-15T13:30:00+00:00,Telekom Cup,FC Bayern München,RB Leipzig,3,1,W,16242805,1786800600,889
1,2026-08-18T16:00:00+00:00,Club Friendly Games,1. FC Heidenheim,FC Bayern München,2,4,W,16849740,1787068800,853
2,2026-08-22T18:30:00+00:00,Supercup,Borussia Dortmund,FC Bayern München,1,2,W,16248441,1787423400,799
3,2026-08-28T18:30:00+00:00,Bundesliga,FC Bayern München,VfB Stuttgart,5,1,W,16434087,1787941800,35
4,2026-09-02T18:45:00+00:00,DFB Pokal,VfL Osnabrück,FC Bayern München,1,4,W,16287047,1788374700,217


Bundesliga form: WDWWW


,date,competition,home_team,away_team,home_score,away_score,result,match_id,timestamp,unique_tournament_id
0,2026-04-25T13:30:00+00:00,Bundesliga,1. FSV Mainz 05,FC Bayern München,3,4,W,14065235,1777123800,35
1,2026-05-02T13:30:00+00:00,Bundesliga,FC Bayern München,1. FC Heidenheim,3,3,D,14065232,1777728600,35
2,2026-05-09T16:30:00+00:00,Bundesliga,VfL Wolfsburg,FC Bayern München,0,1,W,14065251,1778344200,35
3,2026-05-16T13:30:00+00:00,Bundesliga,FC Bayern München,1. FC Köln,5,1,W,14065556,1778938200,35
4,2026-08-28T18:30:00+00:00,Bundesliga,FC Bayern München,VfB Stuttgart,5,1,W,16434087,1787941800,35



VfB Stuttgart (team_id=2677)
Overall form: DWLWL


...

,date,competition,home_team,away_team,home_score,away_score,result,match_id,timestamp,unique_tournament_id
0,2026-04-25T13:30:00+00:00,Bundesliga,FC Augsburg,Eintracht Frankfurt,1,1,D,14065231,1777123800,35
1,2026-05-02T13:30:00+00:00,Bundesliga,SV Werder Bremen,FC Augsburg,1,3,W,14065241,1777728600,35
2,2026-05-09T13:30:00+00:00,Bundesliga,FC Augsburg,Borussia M'gladbach,3,1,W,14065249,1778333400,35
3,2026-05-16T13:30:00+00:00,Bundesliga,1. FC Union Berlin,FC Augsburg,4,0,L,14065558,1778938200,35
4,2026-08-30T15:30:00+00:00,Bundesliga,FC Augsburg,FC Schalke 04,3,0,W,16434039,1788103800,35



FC Schalke 04 (team_id=2530)
Overall form: WLLDL


,date,competition,home_team,away_team,home_score,away_score,result,match_id,timestamp,unique_tournament_id
0,2026-08-01T12:00:00+00:00,Club Friendly Games,KSV Hessen Kassel,FC Schalke 04,0,5,W,16293563,1785585600,853
1,2026-08-08T15:00:00+00:00,Club Friendly Games,FC Schalke 04,Atalanta,0,3,L,16400194,1786201200,853
2,2026-08-16T15:00:00+00:00,Club Friendly Games,FC Schalke 04,Real Madrid,0,3,L,16672725,1786892400,853
3,2026-08-24T18:45:00+00:00,DFB Pokal,Hallescher FC,FC Schalke 04,2,2,D,16287064,1787597100,217
4,2026-08-30T15:30:00+00:00,Bundesliga,FC Augsburg,FC Schalke 04,3,0,L,16434039,1788103800,35


Bundesliga form: L


,date,competition,home_team,away_team,home_score,away_score,result,match_id,timestamp,unique_tournament_id
0,2026-08-30T15:30:00+00:00,Bundesliga,FC Augsburg,FC Schalke 04,3,0,L,16434039,1788103800,35


In [12]:
# 11. Save Timestamp-Keyed JSON Snapshot
snapshot_teams = {
    str(team_id): {
        "team": result["team"],
        "overall_form": result["overall_form"],
        "bundesliga_form": result["bundesliga_form"],
        "overall_matches": result["overall_matches"],
        "bundesliga_matches": result["bundesliga_matches"],
    }
    for team_id, result in form_results.items()
}
current_snapshot = {execution_timestamp: snapshot_teams}

snapshot_json = json.dumps(current_snapshot, ensure_ascii=False, indent=2)
# Handle expected failures with a clear, actionable message.
try:
    snapshot_output_path.write_text(snapshot_json + "\n", encoding="utf-8")
except OSError as exc:
    raise OSError(
        f"Could not save snapshot file {snapshot_output_path}: {exc}"
    ) from exc

print(snapshot_json)


{
  "2026-09-04T10:43:32.643242+02:00": {
    "2672": {
      "team": "FC Bayern München",
      "overall_form": "WWWWW",
      "bundesliga_form": "WDWWW",
      "overall_matches": [
        {
          "match_id": 16242805,
          "date": "2026-08-15T13:30:00+00:00",
          "timestamp": 1786800600,
          "competition": "Telekom Cup",
          "unique_tournament_id": 889,
          "home_team": "FC Bayern München",
          "home_team_id": 2672,
          "away_team": "RB Leipzig",
          "away_team_id": 36360,
          "home_score": 3,
          "away_score": 1,
          "result": "W"
        },
        {
          "match_id": 16849740,
          "date": "2026-08-18T16:00:00+00:00",
          "timestamp": 1787068800,
          "competition": "Club Friendly Games",
          "unique_tournament_id": 853,
          "home_team": "1. FC Heidenheim",
          "home_team_id": 5885,
          "away_team": "FC Bayern München",
          "away_team_id": 2672,
          "home_s

In [13]:
# 12. Close Chrome
if driver is not None:
    # Handle expected failures with a clear, actionable message.
    try:
        driver.quit()
        print("Chrome driver closed.")
    except Exception as exc:
        print(f"Chrome driver shutdown warning: {exc}")
    finally:
        driver = None
else:
    print("Chrome driver is already closed or was not initialized.")


Chrome driver closed.


In [14]:
from project_paths import prune_timestamped_outputs

removed_outputs = prune_timestamped_outputs()
print(f"Pruned {len(removed_outputs)} expired timestamped output(s).")


Pruned 0 expired timestamped output(s).
